In [3]:
#!pip install sql-metadata

In [17]:
from sql_metadata import Parser
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sortedcontainers import SortedDict

In [18]:
def query_parser(query):
    parse_dict = {}

    # lowercasing
    query = query.lower()

    try:
        # Extract tables
        tables = Parser(query).tables
        parse_dict['tables'] = sorted(tables)
    except KeyError as e:
        print(f"KeyError while extracting tables: {e}")
        parse_dict['tables'] = []  # Fallback to an empty list
    except Exception as e:
        print(f"Error while parsing tables: {e}")
        parse_dict['tables'] = []  # Fallback to an empty list
    
    try:
        # Extract columns using sql_metadata parser
        columns = Parser(query).columns
        parse_dict['columns'] = sorted(columns)
    except KeyError as e:
        print(f"KeyError while extracting columns: {e}")
        parse_dict['columns'] = []  # Fallback to an empty list if error occurs
    except Exception as e:
        print(f"Error while parsing columns: {e}")
        parse_dict['columns'] = []  # Fallback to an empty list
    
    try:
        # Extract operations
        operations = Parser(query).columns_dict
        parse_dict['operations'] = SortedDict(operations)
    except KeyError as e:
        print(f"KeyError while extracting operations: {e}")
        parse_dict['operations'] = {}  # Fallback to an empty dictionary
    except Exception as e:
        print(f"Error while parsing operations: {e}")
        parse_dict['operations'] = {}  # Fallback to an empty dictionary
    
    try:
        # Extract tables aliases
        tables_aliases = Parser(query).tables_aliases
        parse_dict['tables_aliases'] = SortedDict(tables_aliases)
    except KeyError as e:
        print(f"KeyError while extracting tables aliases: {e}")
        parse_dict['tables_aliases'] = {}  # Fallback to an empty dictionary
    except Exception as e:
        print(f"Error while parsing tables aliases: {e}")
        parse_dict['tables_aliases'] = {}  # Fallback to an empty dictionary

    return parse_dict

In [19]:
# Function to calculate Jaccard similarity for two sets
def jaccard_similarity(set1, set2):
    if len(set1) > 0 or len(set2) > 0:
        intersection = len(set1.intersection(set2))
        union = len(set1.union(set2))
        return intersection / union if union != 0 else 0
    else:
        return 1

In [20]:
# Function to compare r1 and r2
def compare_queries(r1, r2):
    similarity_score = {}

    # Columns comparison
    columns_r1 = set(r1['columns'])
    columns_r2 = set(r2['columns'])
    similarity_score['columns_similarity'] = jaccard_similarity(columns_r1, columns_r2)

    # Tables comparison
    tables_r1 = set(r1['tables'])
    tables_r2 = set(r2['tables'])
    similarity_score['tables_similarity'] = jaccard_similarity(tables_r1, tables_r2)

    # # Table Aliases comparison
    # aliases_r1 = set(r1['tables_aliases'].keys())
    # aliases_r2 = set(r2['tables_aliases'].keys())
    # similarity_score['aliases_similarity'] = jaccard_similarity(aliases_r1, aliases_r2)

    # Operations comparison (optional, for operation-based similarity)
    operations_r1 = set(r1['operations'].keys()) if r1.get('operations') else set()
    operations_r2 = set(r2['operations'].keys()) if r2.get('operations') else set()
    similarity_score['operations_similarity'] = jaccard_similarity(operations_r1, operations_r2)

    # Overall similarity (average of the individual similarities)
    overall_similarity = sum(similarity_score.values()) / len(similarity_score)
    similarity_score['overall_similarity'] = overall_similarity

    return similarity_score

In [51]:
import pandas as pd
import re

#############################################################################
######### CHANGE INPUT FOLDER FILE HERE #####################################
#############################################################################
file_path = r"C:\Research-Paper\PAPER-WORK-2024\Outputs\Spider-Data-1.xlsx"
# Read the excel file
df = pd.read_excel(file_path, engine='openpyxl')
df.head()

,db_id,spider_query,question,text2sql_query
0,farm,SELECT count(*) FROM farm,How many farms are there?,"SELECT COUNT(*) FROM ""farm"" f;"
1,farm,SELECT count(*) FROM farm,Count the number of farms.,SELECT COUNT(*) FROM farm;
2,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,List the total number of horses on farms in as...,"SELECT f.""Farm_ID"", f.""Total_Horses"" FROM ""far..."
3,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,"What is the total horses record for each farm,...","SELECT f.""Farm_ID"", f.""Total_Horses"" FROM ""far..."
4,farm,SELECT Hosts FROM farm_competition WHERE Theme...,What are the hosts of competitions whose theme...,SELECT DISTINCT fc.Hosts FROM farm_competition...


In [52]:
def get_tables_aliases(parsed_query):
    return parsed_query['tables_aliases']

def normalised_query(query,alias_dict):
    #Lowercasing
    query = query.lower()

    #remove quotes
    query = query.replace('"', '').replace("'", '')
    
    # Iterate through the alias_dict and replace aliases in the query
    for alias, table in alias_dict.items():
        # Use regex to replace the alias with the table name in the query
        query = re.sub(rf'\b{alias}\b', table, query)
    return query    

def detect_hallucination(result1,result2):
    """
    hallucinations in selection
    """ 
    hallucination = False
    if (len(result1) > 0) and (len(result2) > 0):
        if len(set(result1) - set(result2)) > 0:
            hallucination = True
    return hallucination

In [53]:
# Parse queries
df['parsed_spider_query'] = df['spider_query'].apply(query_parser)
df['parsed_text2sql_query'] = df['text2sql_query'].apply(query_parser)

# #################### extract tables_aliases ####################
# df['spider_query_tables_aliases'] = df['parsed_spider_query'].apply(get_tables_aliases)
# df['text2sql_query_tables_aliases'] = df['parsed_text2sql_query'].apply(get_tables_aliases)

#################### Normalise queries ##########################
df['normalised_spider_query'] = df.apply(lambda row: normalised_query(row['spider_query'], get_tables_aliases(row['parsed_spider_query'])), axis=1)
df['normalised_text2sql_query'] = df.apply(lambda row: normalised_query(row['text2sql_query'], get_tables_aliases(row['parsed_text2sql_query'])), axis=1)

#################### Compare the sql queries ####################
df['similarity_score'] = df.apply(lambda row: compare_queries(row['parsed_spider_query'], row['parsed_text2sql_query']), axis=1)



#################### extract table names ####################
df['spider_query_tables'] = df.apply(lambda row: row['parsed_spider_query']['tables'], axis=1)
df['text2sql_query_tables'] = df.apply(lambda row: row['parsed_text2sql_query']['tables'], axis=1)

# Store the tables_similarity score
df['tables_similarity'] = df.apply(lambda row: round(row['similarity_score']['tables_similarity'],2), axis=1)

# hallucinations in table selection
df['hallucinations_in_table_selection'] = df.apply(lambda row: detect_hallucination(row['spider_query_tables'], row['text2sql_query_tables']), axis=1)


#################### extract columns names ####################
df['spider_query_columns'] = df.apply(lambda row: row['parsed_spider_query']['columns'], axis=1)
df['text2sql_query_columns'] = df.apply(lambda row: row['parsed_text2sql_query']['columns'], axis=1)

# Store the columns_similarity score
df['columns_similarity'] = df.apply(lambda row: round(row['similarity_score']['columns_similarity'],2), axis=1)

# hallucinations in column selection
df['hallucinations_in_column_selection'] = df.apply(lambda row: detect_hallucination(row['spider_query_columns'], row['text2sql_query_columns']), axis=1)


#################### extract operations ####################
df['spider_query_operations'] = df.apply(lambda row: row['parsed_spider_query']['operations'], axis=1)
df['text2sql_query_operations'] = df.apply(lambda row: row['parsed_text2sql_query']['operations'], axis=1)

# Store the operations_similarity score
df['operations_similarity'] = df.apply(lambda row: round(row['similarity_score']['operations_similarity'],2), axis=1)

# hallucinations in operations selection
df['hallucinations_in_operations_selection'] = df.apply(lambda row: detect_hallucination(row['spider_query_operations'], row['text2sql_query_operations']), axis=1)


# hallucinations in query
df['hallucinations_in_query'] = df[['hallucinations_in_table_selection','hallucinations_in_column_selection','hallucinations_in_operations_selection']].any(axis=1)

KeyError while extracting columns: 'FROM'
KeyError while extracting operations: 'FROM'
KeyError while extracting columns: 'JOIN'
KeyError while extracting operations: 'JOIN'


In [54]:
df.columns

Index(['db_id', 'spider_query', 'question', 'text2sql_query',
       'parsed_spider_query', 'parsed_text2sql_query',
       'normalised_spider_query', 'normalised_text2sql_query',
       'similarity_score', 'spider_query_tables', 'text2sql_query_tables',
       'tables_similarity', 'hallucinations_in_table_selection',
       'spider_query_columns', 'text2sql_query_columns', 'columns_similarity',
       'hallucinations_in_column_selection', 'spider_query_operations',
       'text2sql_query_operations', 'operations_similarity',
       'hallucinations_in_operations_selection', 'hallucinations_in_query'],
      dtype='object')

In [55]:
df.head()

,db_id,spider_query,question,text2sql_query,parsed_spider_query,parsed_text2sql_query,normalised_spider_query,normalised_text2sql_query,similarity_score,spider_query_tables,...,hallucinations_in_table_selection,spider_query_columns,text2sql_query_columns,columns_similarity,hallucinations_in_column_selection,spider_query_operations,text2sql_query_operations,operations_similarity,hallucinations_in_operations_selection,hallucinations_in_query
0,farm,SELECT count(*) FROM farm,How many farms are there?,"SELECT COUNT(*) FROM ""farm"" f;","{'tables': ['farm'], 'columns': [], 'operation...","{'tables': ['farm'], 'columns': [], 'operation...",select count(*) from farm,select count(*) from farm farm;,"{'columns_similarity': 1, 'tables_similarity':...",[farm],...,False,[],[],1.0,False,{},{},1.0,False,False
1,farm,SELECT count(*) FROM farm,Count the number of farms.,SELECT COUNT(*) FROM farm;,"{'tables': ['farm'], 'columns': [], 'operation...","{'tables': ['farm'], 'columns': [], 'operation...",select count(*) from farm,select count(*) from farm;,"{'columns_similarity': 1, 'tables_similarity':...",[farm],...,False,[],[],1.0,False,{},{},1.0,False,False
2,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,List the total number of horses on farms in as...,"SELECT f.""Farm_ID"", f.""Total_Horses"" FROM ""far...","{'tables': ['farm'], 'columns': ['total_horses...","{'tables': ['farm'], 'columns': ['farm.farm_id...",select total_horses from farm farm total_horse...,"select farm.farm_id, farm.total_horses from fa...","{'columns_similarity': 0.0, 'tables_similarity...",[farm],...,False,[total_horses],"[farm.farm_id, farm.total_horses]",0.0,True,"{'order_by': ['total_horses'], 'select': ['tot...","{'order_by': ['farm.total_horses'], 'select': ...",1.0,False,True
3,farm,SELECT Total_Horses FROM farm ORDER BY Total_H...,"What is the total horses record for each farm,...","SELECT f.""Farm_ID"", f.""Total_Horses"" FROM ""far...","{'tables': ['farm'], 'columns': ['total_horses...","{'tables': ['farm'], 'columns': ['farm.farm_id...",select total_horses from farm farm total_horse...,"select farm.farm_id, farm.total_horses from fa...","{'columns_similarity': 0.0, 'tables_similarity...",[farm],...,False,[total_horses],"[farm.farm_id, farm.total_horses]",0.0,True,"{'order_by': ['total_horses'], 'select': ['tot...","{'order_by': ['farm.total_horses'], 'select': ...",1.0,False,True
4,farm,SELECT Hosts FROM farm_competition WHERE Theme...,What are the hosts of competitions whose theme...,SELECT DISTINCT fc.Hosts FROM farm_competition...,"{'tables': ['farm_competition'], 'columns': ['...","{'tables': ['farm_competition'], 'columns': ['...",select hosts from farm_competition farm_compet...,select distinct farm_competition.hosts from fa...,"{'columns_similarity': 0.0, 'tables_similarity...",[farm_competition],...,False,"[hosts, theme]","[farm_competition.hosts, farm_competition.theme]",0.0,True,"{'select': ['hosts'], 'where': ['theme']}","{'select': ['farm_competition.hosts'], 'where'...",1.0,False,True


In [56]:
#############################################################################
######### CHANGE OUTPUT FOLDER FILE HERE ####################################
#############################################################################
output_file_path = r"C:\Research-Paper\PAPER-WORK-2024\Outputs\Spider-Data-1-Equivalent-Query-Check.xlsx"
df.to_excel(output_file_path, index=False)
print(f"DataFrame saved successfully to {output_file_path}")

DataFrame saved successfully to C:\Research-Paper\PAPER-WORK-2024\Outputs\Spider-Data-1-Equivalent-Query-Check.xlsx
